# Zero-shot evaluation: Malayalam Ayurveda tasks

Evaluates one model at a time on three tasks, with metrics and 95% bootstrap confidence intervals:

| Task | File | What the model does | Main metrics |
|---|---|---|---|
| **qa** | `vaidyamanorama_short_answer_qa_malayalam.csv` | Short-answer questions (closed-book; optional open-book) | exact match, contains-gold, token F1, chrF++ |
| **identification** | `Disease_Identification_Dataset_v1.csv` | Pick 1 of 30 diseases from a symptom description | accuracy, macro-F1 (generation and log-likelihood scoring) |
| **passage_retrieval** | `vaidyamanorama_combined_corpus/queries/relevance.csv` | Find the right passage for a query over a real corpus (embedding model, graded relevance) | recall@k, MRR@10, nDCG@10 |
| *retrieval* (legacy) | `Disease_concept_retrieval_1.csv` | Proxy smoke test only; that file has no passage text. Off by default. | recall@k, MRR, nDCG@10 |

**How to compare models:** change `MODEL_NAME` in the config cell, then *Restart and run all*. Every run saves a JSON and prediction CSVs to `results/`; the last cell builds one comparison table across all runs.

**Notes**
- `FEW_SHOT_K = 0` is pure zero-shot. The same held-out items are scored for every K, so results stay comparable.
- Instruct models use the chat template; base models use a plain completion prompt (`PROMPT_MODE="auto"` picks by checking for a chat template).
- `passage_retrieval` uses an embedding model (`EMBED_MODEL`), independent of `MODEL_NAME`. Run it alone with `TASKS = ["passage_retrieval"]` (no generative model is loaded).

In [1]:
%pip install -q -U transformers accelerate sentence-transformers sacrebleu scikit-learn pandas tqdm
# Larger models in 4-bit: also run  %pip install -q bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 129.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 15.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
dask-cudf-cu13 

In [11]:
from google.colab import userdata
import os
# ===================== CONFIG: normally the only cell you edit =====================
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # needed for gated models (Gemma, Llama); the HF_TOKEN env var also works
MODEL_NAME  = "google/gemma-3-1b-it"   # generative model under test (HF repo id or local path)
EMBED_MODEL = "BAAI/bge-m3"            # embedding model, used only by the retrieval task
TASKS       = ["qa", "identification", "passage_retrieval", "retrieval"]   # also available: "retrieval" (legacy proxy)

FEW_SHOT_K   = 0         # 0 = zero-shot. 1..8 adds that many solved examples to every prompt
PROMPT_MODE  = "auto"    # "auto" | "chat" (instruct models) | "plain" (base models)
DTYPE        = "auto"    # "auto" | "bfloat16" | "float16" | "float32"
LOAD_IN_4BIT = False     # needs bitsandbytes and a GPU
BATCH_SIZE   = 16
MAX_NEW_TOKENS = 48
SCORE_CHUNK  = 10        # candidates scored per forward pass (identification log-likelihood)
HF_TOKEN     = None      # needed for gated models (Gemma, Llama); the HF_TOKEN env var also works

# data
DATA_DIR  = "."
QA_CSV    = "vaidyamanorama_short_answer_qa_malayalam.csv"
IDENT_CSV = "Disease_Identification_Dataset_v1.csv"
RETR_CSV  = "Disease_concept_retrieval_1.csv"          # legacy proxy task only
PR_CORPUS_CSV    = "vaidyamanorama_combined_corpus.csv"     # passage retrieval: passages
PR_QUERIES_CSV   = "vaidyamanorama_combined_queries.csv"    # passage retrieval: queries
PR_RELEVANCE_CSV = "vaidyamanorama_combined_relevance.csv"  # passage retrieval: graded judgements
IDENT_SPLIT    = "test"  # "test" | "validation" | "train" | "all"
QA_CONTEXT_CSV = None    # optional open-book QA: CSV with columns  source_id,text
PASSAGES_CSV   = None    # legacy retrieval only: CSV with columns  passage_id,text  (column names below)
PASSAGE_ID_COL, PASSAGE_TEXT_COL = "passage_id", "text"
QUERY_PREFIX, PASSAGE_PREFIX = "", ""   # e5 models need "query: " and "passage: "
EMBED_MAX_LEN = 1024     # capped automatically at the embedding model's own limit (e5 = 512)
NORMALIZE_TEXT = True    # passage_retrieval headline numbers use normalised text (raw is always reported too)

# misc
SEED = 42
N_BOOT = 1000            # bootstrap resamples for confidence intervals
RESULTS_DIR = "results"
LIMIT = None             # e.g. 10 = quick smoke test on 10 items per task

In [12]:
import os, re, gc, json, glob, time, random, unicodedata, difflib
from collections import Counter
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import f1_score

try:
    from sacrebleu.metrics import CHRF
except Exception:
    CHRF = None
    print("sacrebleu not available: chrF++ will be skipped")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(RESULTS_DIR, exist_ok=True)
if HF_TOKEN or os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=HF_TOKEN or os.environ["HF_TOKEN"])

NEEDS_LM = bool({"qa", "identification"} & set(TASKS))
EMBED_SLUG = EMBED_MODEL.rstrip("/").replace("/", "__")
RUN_NAME = (MODEL_NAME.rstrip("/").replace("/", "__") if NEEDS_LM else f"retrieval-only__{EMBED_SLUG}") + f"__fs{FEW_SHOT_K}"
summary = {"model": MODEL_NAME if NEEDS_LM else None, "embed_model": EMBED_MODEL, "few_shot_k": FEW_SHOT_K,
           "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}

# ---------- Malayalam-aware text normalisation ----------
# Old-style chillu (consonant + virama + ZWJ) -> atomic chillu, then drop ZWJ/ZWNJ.
CHILLU = {"ന്‍": "ൻ", "ര്‍": "ർ", "ല്‍": "ൽ", "ള്‍": "ൾ", "ണ്‍": "ൺ"}
def norm(s):
    s = unicodedata.normalize("NFC", str(s))
    for old, new in CHILLU.items():
        s = s.replace(old, new)
    s = s.replace("‌", "").replace("‍", "")
    # Strip punctuation/symbols by Unicode category. Do NOT use [^\w\s]: Python's \w does not match
    # Malayalam vowel signs (combining marks) and would silently delete parts of words.
    s = "".join(c for c in s if not unicodedata.category(c).startswith(("P", "S")))
    return " ".join(s.lower().split())

def token_f1(pred, gold):
    p, g = pred.split(), gold.split()
    if not p or not g:
        return float(p == g)
    common = sum((Counter(p) & Counter(g)).values())
    if common == 0:
        return 0.0
    pr, rc = common / len(p), common / len(g)
    return 2 * pr * rc / (pr + rc)

def boot_ci(values, n_boot=None, seed=SEED):
    v = np.asarray(values, dtype=float)
    v = v[~np.isnan(v)]
    if len(v) == 0:
        return [float("nan")] * 3
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(v), size=(n_boot or N_BOOT, len(v)))
    means = v[idx].mean(axis=1)
    return [float(v.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))]

def fmt(m):
    return "n/a" if m is None or np.isnan(m[0]) else f"{m[0]*100:.1f}  [{m[1]*100:.1f}, {m[2]*100:.1f}]"

def clean_text(s):
    # for retrieval: NFC, old-style chillu -> atomic chillu, drop ZWJ/ZWNJ, collapse CRLF/whitespace
    s = unicodedata.normalize("NFC", str(s))
    for old, new in CHILLU.items():
        s = s.replace(old, new)
    s = s.replace("‌", "").replace("‍", "")
    return " ".join(s.split())

# ---------- embedding model (shared by the retrieval tasks) ----------
_EMB = {}
def get_embedder():
    if "m" not in _EMB:
        from sentence_transformers import SentenceTransformer
        emb = SentenceTransformer(EMBED_MODEL, device="cuda" if torch.cuda.is_available() else "cpu")
        try:
            limit = emb[0].auto_model.config.max_position_embeddings - 2
        except Exception:
            limit = EMBED_MAX_LEN
        emb.max_seq_length = min(EMBED_MAX_LEN, limit)
        _EMB["m"] = emb
    return _EMB["m"]

def embed_texts(texts, prefix=""):
    return get_embedder().encode([prefix + t for t in texts], normalize_embeddings=True,
                                 batch_size=8, show_progress_bar=False)

def ir_eval(sims, qids, pids, qrels, ks=(1, 3, 5, 10)):
    # sims: (n_queries, n_passages). qrels: {query_id: {passage_id: grade}}; grade > 0 = relevant, 0 = judged non-relevant,
    # anything not listed = unjudged (treated as non-relevant). Returns per-query metric lists and each query's best relevant rank.
    pidx = {p: i for i, p in enumerate(pids)}
    per = {f"recall@{k}": [] for k in ks}
    per.update({"mrr@10": [], "ndcg@10": [], "neg_pairwise_acc": []})
    firsts = []
    for qi, qid in enumerate(qids):
        order = np.argsort(-sims[qi], kind="stable")
        rank_of = np.empty(len(order), dtype=int)
        rank_of[order] = np.arange(1, len(order) + 1)
        judged = qrels.get(qid, {})
        rel = {p: g for p, g in judged.items() if g > 0 and p in pidx}
        assert rel, f"query {qid} has no relevant passage in the corpus"
        rel_ranks = sorted(int(rank_of[pidx[p]]) for p in rel)
        first = rel_ranks[0]
        firsts.append(first)
        for k in ks:
            per[f"recall@{k}"].append(sum(r <= k for r in rel_ranks) / len(rel_ranks))
        per["mrr@10"].append(1.0 / first if first <= 10 else 0.0)
        dcg = sum((2 ** g - 1) / np.log2(rank_of[pidx[p]] + 1) for p, g in rel.items() if rank_of[pidx[p]] <= 10)
        ideal = sorted((2 ** g - 1 for g in rel.values()), reverse=True)[:10]
        idcg = sum(v / np.log2(i + 2) for i, v in enumerate(ideal))
        per["ndcg@10"].append(dcg / idcg)
        negs = [p for p, g in judged.items() if g == 0 and p in pidx]
        per["neg_pairwise_acc"].append(float(np.mean([first < rank_of[pidx[n]] for n in negs])) if negs else np.nan)
    return per, np.array(firsts)
print(HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


None


In [13]:
# ---------- load data ----------
needed = ([QA_CSV] if "qa" in TASKS else []) + ([IDENT_CSV] if "identification" in TASKS else []) \
       + ([RETR_CSV] if "retrieval" in TASKS else []) \
       + ([PR_CORPUS_CSV, PR_QUERIES_CSV, PR_RELEVANCE_CSV] if "passage_retrieval" in TASKS else [])
missing = [f for f in needed if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing:
    try:
        from google.colab import files   # Colab: upload the CSVs when prompted
        print("Please upload:", missing)
        files.upload()
        DATA_DIR = "."
    except ImportError:
        raise FileNotFoundError(f"Missing files in {DATA_DIR!r}: {missing}")

def read(name):
    return pd.read_csv(os.path.join(DATA_DIR, name)).fillna("")

qa = read(QA_CSV) if "qa" in TASKS else None
ident = read(IDENT_CSV) if "identification" in TASKS else None
retr = read(RETR_CSV) if "retrieval" in TASKS else None
assert FEW_SHOT_K <= 8, "FEW_SHOT_K supports 0-8"

if qa is not None:
    # QA: 8 items are reserved as few-shot exemplars and are NEVER scored, so K=0 and K>0 use identical eval items.
    qa_pool = qa.sample(n=8, random_state=SEED)
    qa_eval = qa.drop(qa_pool.index).reset_index(drop=True)
    if LIMIT:
        qa_eval = qa_eval.head(LIMIT)
    print(f"QA eval items: {len(qa_eval)}")

if ident is not None:
    # Identification: exemplars (if any) come from the train split; eval defaults to the test split.
    LABELS = sorted(ident.disease_label.unique())
    ident_shots = ident[ident.split == "train"].sample(n=FEW_SHOT_K, random_state=SEED) if FEW_SHOT_K else ident.iloc[0:0]
    ident_eval = ident if IDENT_SPLIT == "all" else ident[ident.split == IDENT_SPLIT]
    ident_eval = ident_eval.drop(ident_shots.index, errors="ignore").reset_index(drop=True)
    if LIMIT:
        ident_eval = ident_eval.head(LIMIT)
    print(f"Identification eval items ({IDENT_SPLIT}): {len(ident_eval)} across {len(LABELS)} labels")
else:
    LABELS = []   # keeps the prompt definitions below valid when the task is not selected

if retr is not None and LIMIT:
    retr = retr.head(LIMIT)

QA eval items: 141
Identification eval items (test): 90 across 30 labels


## Load the generative model
Skipped automatically if `TASKS` contains only `retrieval`.

In [14]:
for _n in ("model",):                      # free memory when switching models in the same session
    if _n in globals():
        del globals()[_n]
gc.collect()
if torch.cuda.is_available():

    torch.cuda.empty_cache()

def pick_dtype():
    if DTYPE != "auto":
        return getattr(torch, DTYPE)
    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
        return torch.bfloat16
    # T4 and CPU: float16 can overflow on some models (notably Gemma), so default to float32
    return torch.float32

if NEEDS_LM:
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kw = {}
    ver = tuple(int(x) for x in re.findall(r"\d+", transformers.__version__)[:2])
    kw["dtype" if ver >= (4, 56) else "torch_dtype"] = pick_dtype()
    if LOAD_IN_4BIT:
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=pick_dtype())
    if torch.cuda.is_available():
        kw["device_map"] = "auto"
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kw).eval()
    if not torch.cuda.is_available():
        model.to("cpu")

    MODE = PROMPT_MODE if PROMPT_MODE != "auto" else ("chat" if getattr(tok, "chat_template", None) else "plain")
    ADD_SPECIAL = (MODE == "plain")   # chat templates already include BOS where the model needs it
    summary.update({"prompt_mode": MODE, "dtype": str(pick_dtype()).replace("torch.", ""),
                    "n_params_M": round(sum(p.numel() for p in model.parameters()) / 1e6, 1)})
    print(f"Loaded {MODEL_NAME} | prompt mode: {MODE} | dtype: {summary['dtype']} | params: {summary['n_params_M']}M")

def render(user_text, plain_suffix):
    if MODE == "chat":
        return tok.apply_chat_template([{"role": "user", "content": user_text}], tokenize=False,
                                       add_generation_prompt=True, enable_thinking=False)
    return user_text + "\n" + plain_suffix

@torch.no_grad()
def generate_batch(prompts, max_new_tokens=None, batch_size=None):
    max_new_tokens, batch_size = max_new_tokens or MAX_NEW_TOKENS, batch_size or BATCH_SIZE
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))   # similar lengths batch together
    outs = [None] * len(prompts)
    for s in tqdm(range(0, len(order), batch_size), desc="generating"):
        ids = order[s:s + batch_size]
        enc = tok([prompts[i] for i in ids], return_tensors="pt", padding=True, add_special_tokens=ADD_SPECIAL)
        enc.pop("token_type_ids", None)
        enc = enc.to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             temperature=None, top_p=None, top_k=None, pad_token_id=tok.pad_token_id)
        texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        for i, t in zip(ids, texts):
            outs[i] = t
    return outs

_PREFIX = re.compile(r"^\s*(answer|disease|ഉത്തരം|രോഗം)\s*[:：\-]\s*", re.I)
def clean(text):
    t = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    lines = [l.strip() for l in t.splitlines() if l.strip()]
    t = lines[0] if lines else ""
    return _PREFIX.sub("", t).strip().strip("\"'“”‘’*`").strip()

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.00GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Loaded google/gemma-3-1b-it | prompt mode: chat | dtype: float32 | params: 999.9M


## Tokenizer fertility
Tokens per word on your own text. Lower is better: it means more context fits, training uses less memory, and words are not shattered into fragments.

In [15]:
def fertility(texts):
    texts = [t for t in texts if str(t).strip()]
    toks = sum(len(tok.encode(t, add_special_tokens=False)) for t in texts)
    words = sum(len(t.split()) for t in texts)
    chars = sum(len(t) for t in texts)
    return {"tokens_per_word": round(toks / words, 2), "chars_per_token": round(chars / toks, 2)}

if NEEDS_LM:
    fert = {}
    if qa is not None:
        fert.update({"qa_questions": fertility(qa.question), "qa_answers": fertility(qa.answer)})
    if ident is not None:
        fert["identification_labels"] = fertility(LABELS)
    if "passage_retrieval" in TASKS:
        fert["passage_queries"] = fertility(read(PR_QUERIES_CSV).query_text_ml)
    summary["fertility"] = fert
    print(pd.DataFrame(fert).T.to_string())

                       tokens_per_word  chars_per_token
qa_questions                      3.60             2.61
qa_answers                        4.03             1.99
identification_labels             4.10             1.83
passage_queries                   3.37             3.10


## Task 1: short-answer QA (closed-book)
Metrics per item, averaged with a 95% bootstrap CI:
- **em**: exact match after normalisation; **em_unordered**: same, ignoring word order (for multi-item answers like "ശ്വാസം, കാസം")
- **contains_gold**: the gold answer appears inside the prediction
- **token_f1**: word-overlap F1; **chrf**: mean sentence chrF++ (character n-grams, good for Malayalam morphology)
- **empty**: share of empty outputs (a model-failure signal, not a quality score)

A constant "majority answer" baseline is reported for reference. Any model should clear it.

In [16]:
QA_INSTR = ("You are an expert in Ayurveda, answering questions based on the Malayalam text Vaidyamanorama. "
            "Reply in Malayalam with only the answer: a single word or short phrase, no explanation.")

def qa_user_text(q, ctx=None, shots=()):
    parts = [QA_INSTR, ""]
    for sq, sa in shots:
        parts += [f"Question: {sq}", f"Answer: {sa}", ""]
    if ctx:
        parts += [f"Passage: {ctx}", ""]
    parts += [f"Question: {q}"]
    return "\n".join(parts)

_chrf = CHRF(word_order=2) if CHRF else None
def qa_item_scores(pred, gold):
    p, g = norm(pred), norm(gold)
    return {"em": float(p == g),
            "em_unordered": float(sorted(p.split()) == sorted(g.split())),
            "contains_gold": float(bool(g) and g in p),
            "token_f1": token_f1(p, g),
            "chrf": _chrf.sentence_score(p, [g]).score / 100 if _chrf else np.nan,
            "empty": float(p == "")}

QA_METRICS = ["em", "em_unordered", "contains_gold", "token_f1", "chrf", "empty"]

def run_qa(df_eval, ctx_map, tag):
    shots = [(r.question, r.answer) for r in qa_pool.head(FEW_SHOT_K).itertuples()]
    rows = df_eval[df_eval.source_id.isin(ctx_map)] if ctx_map else df_eval
    prompts = [render(qa_user_text(r.question, ctx_map.get(r.source_id) if ctx_map else None, shots), "Answer:")
               for r in rows.itertuples()]
    raw = generate_batch(prompts)
    out = rows.reset_index(drop=True).copy()
    out["raw"], out["pred"] = raw, [clean(t) for t in raw]
    out = pd.concat([out, pd.DataFrame([qa_item_scores(p, g) for p, g in zip(out.pred, out.answer)])], axis=1)
    maj = out.answer.map(norm).value_counts().index[0]
    res = {"n": len(out), "metrics": {m: boot_ci(out[m]) for m in QA_METRICS},
           "majority_baseline_em": float((out.answer.map(norm) == maj).mean())}
    out.to_csv(f"{RESULTS_DIR}/{RUN_NAME}__qa_{tag}_predictions.csv", index=False, encoding="utf-8")
    print(f"\n=== QA ({tag}), n={res['n']} ===")
    for m in QA_METRICS:
        print(f"{m:>14}: {fmt(res['metrics'][m])}")
    print(f"{'majority-answer EM baseline':>28}: {res['majority_baseline_em']*100:.1f}")
    return res, out

if "qa" in TASKS:
    summary["qa"] = {}
    summary["qa"]["closed_book"], qa_out = run_qa(qa_eval, None, "closed_book")
    if QA_CONTEXT_CSV:
        c = read(QA_CONTEXT_CSV)
        summary["qa"]["open_book"], _ = run_qa(qa_eval, dict(zip(c["source_id"], c["text"])), "open_book")

    by_ch = qa_out.groupby("chapter").agg(n=("em", "size"), em=("em", "mean"), token_f1=("token_f1", "mean"))
    print("\nPer-chapter (chapters with n < 5 are noisy):")
    print(by_ch.sort_values("n", ascending=False).round(2).to_string())
    print("\nSample of wrong answers:")
    wrong = qa_out[qa_out.em == 0][["question", "answer", "pred"]]
    print(wrong.sample(min(12, len(wrong)), random_state=SEED).to_string(index=False))

generating:   0%|          | 0/9 [00:00<?, ?it/s]


=== QA (closed_book), n=141 ===
            em: 0.0  [0.0, 0.0]
  em_unordered: 0.0  [0.0, 0.0]
 contains_gold: 0.0  [0.0, 0.0]
      token_f1: 0.0  [0.0, 0.0]
          chrf: 0.4  [0.2, 0.6]
         empty: 0.0  [0.0, 0.0]
 majority-answer EM baseline: 2.1

Per-chapter (chapters with n < 5 are noisy):
                                        n   em  token_f1
chapter                                                 
ജ്വരം                                  14  0.0       0.0
ഉന്മാദം, അപസ്മാരം                      10  0.0       0.0
രക്തപിത്തം                              8  0.0       0.0
നേത്രരോഗം                               7  0.0       0.0
മൂത്രകൃച്ഛ്രം                           7  0.0       0.0
കുഷ്ഠം                                  6  0.0       0.0
ഛർദ്ദി, തൃഷ്ണ                           6  0.0       0.0
അതിസാരം                                 6  0.0       0.0
കാസം                                    5  0.0       0.0
പ്രതിശ്യായം, പീനസം, ശിരോരോഗം, ഗ്രന്ഥി   5  0.0       0.0
ഹിക്കാശ്വാസ

## Task 2: disease identification (30-way classification)
The prompt lists all 30 labels. Two ways to read the model's answer:
- **Generation**: the model writes a label, which is mapped to the nearest valid label. Unmappable output counts as wrong (`invalid` rate reported).
- **Log-likelihood scoring**: the model scores every label as a continuation of the prompt and the highest wins, so output formatting can't cause failures. This is the standard method for classification with small models. `ll_char` normalises by label length in characters (like lm-eval's `acc_norm`), `ll_tok` per token, `ll_sum` is raw.

Chance level is 1/30 = 3.3%. Note that these descriptions are templated and recombine the same symptom phrases per disease, so scores may saturate quickly after fine-tuning.

In [ ]:
IDENT_INSTR = ("You are an expert in Ayurveda. Below is a clinical description of a patient. Choose the single most "
               "likely disease from the list. Reply with the disease name exactly as written in the list, nothing else.\n\n"
               "Diseases:\n" + "\n".join(f"- {l}" for l in LABELS))

def ident_user_text(txt, shots=()):
    parts = [IDENT_INSTR, ""]
    for st, sl in shots:
        parts += [f"Description: {st}", f"Disease: {sl}", ""]
    parts += [f"Description: {txt}"]
    return "\n".join(parts)

NORM_LABELS = {norm(l): l for l in LABELS}
def map_label(text):
    n = norm(text)
    if n in NORM_LABELS:
        return NORM_LABELS[n], "exact"
    hits = [l for nl, l in NORM_LABELS.items() if nl and nl in n]
    if hits:
        return max(hits, key=lambda l: len(norm(l))), "contains"
    m = difflib.get_close_matches(n, list(NORM_LABELS), n=1, cutoff=0.75)
    if m:
        return NORM_LABELS[m[0]], "fuzzy"
    return "INVALID", "invalid"

@torch.no_grad()
def candidate_logprobs(prompt_ids, cands, chunk=None):
    # sum of log P(candidate tokens | prompt) for each candidate; left-padded, only needed logits kept
    chunk, res, dev = chunk or SCORE_CHUNK, [], model.device
    for s in range(0, len(cands), chunk):
        group = cands[s:s + chunk]
        seqs = [prompt_ids + c for c in group]
        L, keep = max(len(x) for x in seqs), max(len(c) for c in group) + 1
        ids = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long)
        mask = torch.zeros((len(seqs), L), dtype=torch.long)
        for j, x in enumerate(seqs):
            ids[j, L - len(x):] = torch.tensor(x)
            mask[j, L - len(x):] = 1
        pos = (mask.cumsum(-1) - 1).clamp(min=0)
        ids, mask, pos = ids.to(dev), mask.to(dev), pos.to(dev)
        try:
            logits = model(input_ids=ids, attention_mask=mask, position_ids=pos, logits_to_keep=keep).logits
        except TypeError:   # older transformers without logits_to_keep
            logits = model(input_ids=ids, attention_mask=mask, position_ids=pos).logits[:, -keep:]
        logp = torch.log_softmax(logits[:, :keep - 1].float(), dim=-1)
        tok_lp = logp.gather(-1, ids[:, L - keep + 1:].unsqueeze(-1)).squeeze(-1)
        lens = torch.tensor([len(c) for c in group], device=dev)
        valid = torch.arange(keep - 1, device=dev).unsqueeze(0) >= (keep - 1 - lens).unsqueeze(1)
        res.extend((tok_lp * valid).sum(-1).tolist())
    return res

def boot_macro_f1(pred, y, n_boot=None):
    n_boot = min(n_boot or N_BOOT, 500)
    rng = np.random.default_rng(SEED)
    y, p = np.array(y, dtype=object), np.array(pred, dtype=object)
    vals = [f1_score(y[i], p[i], labels=sorted(set(y[i])), average="macro", zero_division=0)
            for i in rng.integers(0, len(y), size=(n_boot, len(y)))]
    return [float(f1_score(y, p, labels=sorted(set(y)), average="macro", zero_division=0)),
            float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))]

if "identification" in TASKS:
    shots = list(zip(ident_shots.input_text, ident_shots.disease_label))
    users = [ident_user_text(t, shots) for t in ident_eval.input_text]
    prompts = [render(u, "Disease:") for u in users]
    y = list(ident_eval.disease_label)
    out = ident_eval.copy()

    # --- generation ---
    raw = generate_batch(prompts)
    mapped = [map_label(clean(t)) for t in raw]
    out["raw"], out["pred_gen"], out["match"] = raw, [m[0] for m in mapped], [m[1] for m in mapped]

    # --- log-likelihood scoring ---
    cand = {l: tok.encode((" " if MODE == "plain" else "") + l, add_special_tokens=False) for l in LABELS}
    cand_ids = [cand[l] for l in LABELS]
    n_chars = np.array([len(l) for l in LABELS])
    n_toks = np.array([len(c) for c in cand_ids])
    ll = []
    for p in tqdm(prompts, desc="scoring labels"):
        pid = tok(p, add_special_tokens=ADD_SPECIAL)["input_ids"]
        ll.append(candidate_logprobs(pid, cand_ids))
    ll = np.array(ll)                                   # (items, labels)
    out["pred_ll_sum"] = [LABELS[i] for i in ll.argmax(1)]
    out["pred_ll_tok"] = [LABELS[i] for i in (ll / n_toks).argmax(1)]
    out["pred_ll_char"] = [LABELS[i] for i in (ll / n_chars).argmax(1)]

    methods = {"gen": "pred_gen", "ll_sum": "pred_ll_sum", "ll_tok": "pred_ll_tok", "ll_char": "pred_ll_char"}
    res = {"n": len(out), "split": IDENT_SPLIT, "chance": 1 / len(LABELS),
           "invalid_rate": float((out.match == "invalid").mean()),
           "match_types": out.match.value_counts().to_dict(), "accuracy": {}, "macro_f1": {}}
    for k, col in methods.items():
        res["accuracy"][k] = boot_ci([float(a == b) for a, b in zip(out[col], y)])
        res["macro_f1"][k] = boot_macro_f1(list(out[col]), y)
    summary["identification"] = res
    out.to_csv(f"{RESULTS_DIR}/{RUN_NAME}__identification_predictions.csv", index=False, encoding="utf-8")

    print(f"=== Identification ({IDENT_SPLIT}), n={res['n']}, chance={res['chance']*100:.1f}% ===")
    print(pd.DataFrame({"accuracy": {k: fmt(v) for k, v in res["accuracy"].items()},
                        "macro_f1": {k: fmt(v) for k, v in res["macro_f1"].items()}}).to_string())
    print(f"generation: invalid output rate {res['invalid_rate']*100:.1f}%, match types {res['match_types']}")
    best = max(methods, key=lambda k: res["accuracy"][k][0])
    conf = Counter((a, b) for a, b in zip(y, out[methods[best]]) if a != b).most_common(8)
    print(f"\nMost frequent confusions with '{best}' (true -> predicted):")
    for (a, b), c in conf:
        print(f"  {c}x  {a} -> {b}")

generating:   0%|          | 0/6 [00:00<?, ?it/s]

scoring labels:   0%|          | 0/90 [00:00<?, ?it/s]

## Task 3: passage retrieval (combined corpus)
`EMBED_MODEL` embeds the passages and queries, and passages are ranked by cosine similarity. Judgements are graded: 2 = relevant, 0 = judged non-relevant, not listed = unjudged (counted as non-relevant).

Metrics: `recall@1/3/5/10` (with exactly one relevant passage per query this equals the hit rate), `mrr@10`, `ndcg@10` (gain 2^grade − 1), and `neg_pairwise_acc` (how often the relevant passage outranks the judged negative).

Four variants are reported so you can see what drives the result:
- `embed_raw` / `embed_clean`: the embedding model on the text as stored vs. normalised (old-style ZWJ chillu converted, line breaks collapsed). The passages are OCR text with ZWJ-encoded chillu while the queries use standard encoding, so this can matter a lot.
- `tfidf_raw` / `tfidf_clean`: a character n-gram TF-IDF baseline with no model, to show how much the neural embedder adds.

**Read the numbers with these caveats**
- The corpus has only as many passages as queries, one gold passage each. That is far easier than searching a whole book, so scores will be high and models will look closer together than they really are.
- Some queries refer to "this page" (പേജ്/പേജിൽ), which no retriever can resolve. The code counts them and breaks them out below.
- All judgements are marked `pending_review` and the text is unverified OCR. Unjudged passages count as non-relevant, so a retriever can be penalised for returning a passage that is actually relevant.

In [ ]:
if "passage_retrieval" in TASKS:
    from sklearn.feature_extraction.text import TfidfVectorizer
    corpus, pq, rel = read(PR_CORPUS_CSV), read(PR_QUERIES_CSV), read(PR_RELEVANCE_CSV)
    pids = corpus.passage_id.tolist()
    pid_set = set(pids)

    # graded judgements: the relevance file first; the gold lists in the queries file fill any gap
    qrels = {}
    for r_ in rel.itertuples():
        qrels.setdefault(r_.query_id, {})[r_.passage_id] = int(r_.relevance_grade)
    for r_ in pq.itertuples():
        for g, lv in zip(json.loads(r_.gold_passage_ids), json.loads(r_.gold_relevance_levels)):
            qrels.setdefault(r_.query_id, {}).setdefault(g, int(lv))
    keep = [any(g > 0 and p in pid_set for p, g in qrels.get(qid, {}).items()) for qid in pq.query_id]
    if not all(keep):
        print(f"{len(keep) - sum(keep)} queries dropped: no relevant passage in the corpus")
    pq = pq[keep].reset_index(drop=True)
    if LIMIT:
        pq = pq.head(LIMIT)
    qids = pq.query_id.tolist()
    n_page_refs = int(pq.query_text_ml.str.contains("പേജ").sum())
    print(f"{len(qids)} queries over {len(pids)} passages | queries that refer to 'this page': {n_page_refs} "
          f"| judged negatives: {sum(g == 0 for q_ in qids for g in qrels[q_].values())}")

    raw_q, raw_p = pq.query_text_ml.tolist(), corpus.passage_text_ml.tolist()
    texts = {"raw": (raw_q, raw_p), "clean": ([clean_text(t) for t in raw_q], [clean_text(t) for t in raw_p])}
    sims = {}
    for vname, (qs, ps) in texts.items():
        Dm, Qm = embed_texts(ps, PASSAGE_PREFIX), embed_texts(qs, QUERY_PREFIX)
        sims[f"embed_{vname}"] = Qm @ Dm.T
        tf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True).fit(ps)
        sims[f"tfidf_{vname}"] = (tf.transform(qs) @ tf.transform(ps).T).toarray()

    emb = get_embedder()
    ntok = [len(emb.tokenizer(t)["input_ids"]) for t in texts["clean"][1]]
    n_trunc = int(sum(n > emb.max_seq_length for n in ntok))
    print(f"embedding max_seq_length = {emb.max_seq_length}; passages longer than that (truncated): {n_trunc} of {len(ntok)} "
          f"(median {int(np.median(ntok))} tokens, max {max(ntok)})")

    results, firsts = {}, {}
    for name, S in sims.items():
        per, first = ir_eval(S, qids, pids, qrels)
        results[name] = {m: boot_ci(v) for m, v in per.items()}
        firsts[name] = first
    primary = "embed_clean" if NORMALIZE_TEXT else "embed_raw"
    summary["passage_retrieval"] = {"n_queries": len(qids), "n_passages": len(pids), "primary": primary,
                                    "embed_model": EMBED_MODEL, "max_seq_length": int(emb.max_seq_length),
                                    "truncated_passages": n_trunc, "queries_referring_to_page": n_page_refs,
                                    "variants": results}

    print(f"\n=== Passage retrieval: {EMBED_MODEL} (headline variant: {primary}) ===")
    print(pd.DataFrame({n: {m: fmt(v) for m, v in r.items()} for n, r in results.items()}).T.to_string())

    meta = pq.assign(first_rank=firsts[primary], mentions_page=pq.query_text_ml.str.contains("പേജ"),
                     split_=pq.split.replace("", "(none)"))
    def grp(col):
        g = meta.groupby(col).first_rank
        return pd.DataFrame({"n": g.size(), "recall@1": g.apply(lambda r: (r <= 1).mean()),
                             "recall@5": g.apply(lambda r: (r <= 5).mean()),
                             "mrr@10": g.apply(lambda r: np.where(r <= 10, 1.0 / r, 0.0).mean())}).round(2)
    for col in ("query_generation_method", "mentions_page", "difficulty", "query_type", "split_"):
        print(f"\nBy {col}:\n{grp(col).to_string()}")

    top1 = sims[primary].argmax(1)
    miss = [i for i in range(len(qids)) if meta.first_rank[i] > 1]
    print(f"\nQueries where the relevant passage is not ranked first ({len(miss)}), first 15:")
    for i in miss[:15]:
        print(f"  {qids[i]}  relevant rank {meta.first_rank[i]:>2} | query topic: {pq.query_topic_ml[i]} "
              f"| returned: {pids[top1[i]]} ({corpus.passage_topic_ml[top1[i]]})")

    out = pd.DataFrame({"query_id": qids, "split": pq.split, "query_generation_method": pq.query_generation_method,
                        "top1_" + primary: [pids[j] for j in top1]})
    for name in sims:
        out["rank_" + name] = firsts[name]
    out.to_csv(f"{RESULTS_DIR}/{RUN_NAME}__passage_retrieval_predictions.csv", index=False, encoding="utf-8")

## Legacy task: proxy retrieval (off by default)
Uses `Disease_concept_retrieval_1.csv`, which has no passage text. Without `PASSAGES_CSV` the corpus is built from the file's own `answer` descriptions (17 documents that paraphrase the queries), so scores are inflated and not comparable to anything. It only checks that the pipeline runs. Use `passage_retrieval` above for a real benchmark; add `"retrieval"` to `TASKS` to run this one.

In [ ]:
if "retrieval" in TASKS:
    proxy = PASSAGES_CSV is None
    if proxy:
        print("WARNING: PROXY MODE. Corpus is built from the 'answer' descriptions (17 docs), so scores are inflated.")
        docs = retr.groupby("relevant_passage_id")["answer"].apply(lambda s: " ".join(dict.fromkeys(s))).reset_index()
        docs.columns = ["passage_id", "text"]
    else:
        docs = read(PASSAGES_CSV).rename(columns={PASSAGE_ID_COL: "passage_id", PASSAGE_TEXT_COL: "text"})[["passage_id", "text"]]
    known = set(docs.passage_id)
    q = retr[retr.relevant_passage_id.isin(known)].reset_index(drop=True)
    if len(q) < len(retr):
        print(f"{len(retr) - len(q)} queries dropped: gold passage id not found in corpus")

    D = embed_texts(docs.text, PASSAGE_PREFIX)
    Q = embed_texts(q.question, QUERY_PREFIX)
    sims = Q @ D.T
    ids = docs.passage_id.to_numpy()
    gold_idx = np.array([int(np.where(ids == g)[0][0]) for g in q.relevant_passage_id])
    gold_sim = sims[np.arange(len(q)), gold_idx]
    rank = 1 + (sims > gold_sim[:, None]).sum(1)          # 1 = gold passage ranked first

    per_item = {f"recall@{k}": (rank <= k).astype(float) for k in (1, 3, 5, 10)}
    per_item["mrr@10"] = np.where(rank <= 10, 1.0 / rank, 0.0)
    per_item["ndcg@10"] = np.where(rank <= 10, 1.0 / np.log2(rank + 1), 0.0)
    res = {"proxy_mode": proxy, "n_queries": len(q), "corpus_size": len(docs),
           "metrics": {m: boot_ci(v) for m, v in per_item.items()}}
    summary["retrieval"] = res

    out = q.copy()
    out["gold_rank"] = rank
    out["top5"] = [" | ".join(ids[np.argsort(-s)[:5]]) for s in sims]
    out.to_csv(f"{RESULTS_DIR}/{RUN_NAME}__retrieval_predictions.csv", index=False, encoding="utf-8")

    print(f"=== Retrieval ({'PROXY' if proxy else 'real'}), {len(q)} queries, corpus size {len(docs)} ===")
    for m, v in res["metrics"].items():
        print(f"{m:>10}: {fmt(v)}")
    for col in ("difficulty", "query_type"):
        tab = pd.DataFrame({"n": out.groupby(col).size(), "recall@1": out.groupby(col).gold_rank.apply(lambda r: (r <= 1).mean()),
                            "recall@5": out.groupby(col).gold_rank.apply(lambda r: (r <= 5).mean())})
        print(f"\nBy {col}:\n{tab.round(2).to_string()}")

## Save this run

In [ ]:
path = f"{RESULTS_DIR}/{RUN_NAME}.json"
with open(path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print("Saved", path)

## Compare all runs
Reads every JSON in `results/` (one per model/K). Values are `mean [95% CI]` in percent. With small test sets, treat gaps smaller than the interval width as noise.

In [ ]:
def pick(d, *keys):
    for k in keys:
        if not isinstance(d, dict) or k not in d:
            return None
        d = d[k]
    return d

def pr_metric(s, m):
    p = s.get("passage_retrieval")
    return fmt(pick(p, "variants", p["primary"], m)) if p else "n/a"

rows = []
for fpath in sorted(glob.glob(f"{RESULTS_DIR}/*.json")):
    s = json.load(open(fpath, encoding="utf-8"))
    rows.append({"model": s.get("model") or "(retrieval only)", "K": s.get("few_shot_k"), "mode": s.get("prompt_mode"),
                 "params(M)": s.get("n_params_M"),
                 "tok/word (QA q)": pick(s, "fertility", "qa_questions", "tokens_per_word"),
                 "QA EM": fmt(pick(s, "qa", "closed_book", "metrics", "em")),
                 "QA F1": fmt(pick(s, "qa", "closed_book", "metrics", "token_f1")),
                 "QA chrF++": fmt(pick(s, "qa", "closed_book", "metrics", "chrf")),
                 "Ident acc (gen)": fmt(pick(s, "identification", "accuracy", "gen")),
                 "Ident acc (ll_char)": fmt(pick(s, "identification", "accuracy", "ll_char")),
                 "Ident macro-F1 (ll_char)": fmt(pick(s, "identification", "macro_f1", "ll_char")),
                 "PassRetr R@1": pr_metric(s, "recall@1"),
                 "PassRetr R@5": pr_metric(s, "recall@5"),
                 "PassRetr MRR@10": pr_metric(s, "mrr@10"),
                 "PassRetr nDCG@10": pr_metric(s, "ndcg@10"),
                 "TF-IDF R@1 (baseline)": fmt(pick(s, "passage_retrieval", "variants", "tfidf_clean", "recall@1")),
                 "Embed model": s["embed_model"] if s.get("passage_retrieval") else None})
comparison = pd.DataFrame(rows)
pd.set_option("display.width", 250, "display.max_columns", 30, "display.max_colwidth", 40)
comparison